# 질문 → Cypher → 프로젝트 Aura 그래프 시연

운영 검색 Tool·메인 LangGraph와 분리된 개발자용 시연입니다. **현재 사용자의 요청에 따라 기존 계정을 사용합니다. 계정 자체가 읽기 전용이라는 뜻은 아닙니다.** 프로그램의 단일 문장·프로시저/외부 접근 검사, EXPLAIN의 `query_type == 'r'`, 실행 직전 재검사, 읽기 트랜잭션·10초 제한을 적용합니다. DB 데이터·스키마·계정 권한을 변경하지 않습니다.

1. 프로젝트 Aura와 실제 스키마·값 사전을 확인합니다.
2. 자유 질문을 입력하고 **생성 → 검사 → 실행** 버튼을 순서대로 누릅니다.
3. 그래프 요청은 Path를 보존합니다. 복사란의 **동일한 Cypher**를 같은 Aura의 Query/Browser 화면에 붙여 그래프 보기로 확인합니다.
4. 집계는 표로 확인합니다. 결과는 Citation을 대신하지 않습니다.

Chrome 자동 제어가 승인되지 않아 Aura 화면 대조는 아직 미확인입니다. 아래 마지막 확인란은 사용자가 실제 화면을 확인한 뒤에만 체크합니다. 실제 결과·한계는 `text2cypher_eval_summary.json`을 따릅니다.

In [1]:
import hashlib
import json
import sys
from html import escape
from pathlib import Path

import ipywidgets as widgets
from IPython.display import JSON, display

here = Path.cwd().resolve()
project_root = next(p for p in [here, *here.parents] if (p / "src/rag/respiratory").is_dir())
folder = project_root / "notebooks/respiratory"
for path in (project_root / "src", folder):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
import text2cypher_demo as demo

print("설치된 패키지 버전:", json.dumps(demo.versions(), ensure_ascii=False))

설치된 패키지 버전: {"neo4j": "6.4.1", "langchain-openai": "1.7.3", "neo4j-graphrag": null, "ipywidgets": "8.1.9", "nbformat": "5.11.1"}


## 1. 프로젝트 연결 및 접근 범위

설정은 프로젝트 `.env`에서 읽으며 URI·계정·비밀번호·API 키를 출력하지 않습니다. 기존 URI는 Aura TLS 주소 그대로 사용합니다. `NEO4J_READONLY_USERNAME`·`NEO4J_READONLY_PASSWORD`를 따로 설정하면 해당 계정을 우선 사용합니다. 역할을 확인할 수 없거나 관리자용 계정이어도 읽기 전용 계정이라고 표시하지 않습니다.

접근 범위는 **프로젝트 Aura의 현재 Document 전체**입니다. 문서별 접근 제어를 프롬프트로 구현했다고 주장하지 않습니다. 별도 로컬 DB를 만들거나 데이터를 재적재하지 않습니다.

In [2]:
ready = False
if globals().get("driver") is not None:
    globals()["driver"].close()
driver = None
schema = None
llm = None
try:
    driver, database, connection = demo.connect_project()
    schema = demo.collect_schema(driver, database, connection)
    schema["account"]["user_direction"] = "11/9 사용자 요청: 기존 계정으로 프로그램에서 읽기만 수행"
    demo.SCHEMA_PATH.write_text(json.dumps(schema, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    llm = demo.make_llm()
    connected = demo._metadata_rows(driver, database, "RETURN 1 AS connected")[0]["connected"]
    ready = connected == 1
    display(JSON({"connection": connection, "connected": connected,
                  "account": schema["account"], "scope": schema["scope"],
                  "documents": schema["dictionaries"]["Document"]}))
except Exception as error:  # noqa: BLE001 -- 비밀이 섞인 오류 원문 대신 종류만 표시
    display(JSON({"status": "CONNECTION_ERROR", "error": demo.safe_error(error)}))

<IPython.core.display.JSON object>

## 2. LLM이 참고하는 실제 스키마·값 사전

아래 내용은 명세를 그대로 복사한 것이 아니라 실제 DB에서 관찰한 구조와 값 사전에 의미 규칙을 합친 것입니다. `Chunk.embedding` 값과 전체 원문은 수집·전송하지 않습니다. 속성 목록은 노드별 필수 속성 목록이 아닌 관찰된 속성의 합집합입니다. 기존 고정 SCHEMA에 있던 `ActionConcept.group`의 실제 존재 여부도 차이표에 기록합니다.

In [3]:
if schema:
    sections = [
        ("구조·관계 방향", {"properties": schema["properties"], "relationships": schema["relationships"]}),
        ("문서·질병·역할·행동 개념 값 사전", schema["dictionaries"]),
        ("의미 규칙·명세 차이", {"semantics": schema["semantics"], "differences": schema["differences"]}),
    ]
    panels = [widgets.HTML("<pre style='white-space:pre-wrap'>" + escape(json.dumps(value, ensure_ascii=False, indent=2)) + "</pre>") for _, value in sections]
    accordion = widgets.Accordion(children=panels)
    for i, (title, _) in enumerate(sections):
        accordion.set_title(i, title)
    display(accordion)
else:
    print("연결 셀을 먼저 실행하세요.")

Accordion(children=(HTML(value="<pre style='white-space:pre-wrap'>{\n  &quot;properties&quot;: {\n    &quot;Do…

## 3. 자유 입력 → 생성 → 검사 → 실행

**생성 버튼은 LLM만 호출합니다. 검사 버튼은 EXPLAIN까지만 실행합니다. 실행 버튼은 검사된 동일 쿼리만 실행합니다.** 쿼리나 질문을 편집하면 이전 검사는 무효가 됩니다. 원래 생성문과 수정·실행문을 각각 보존합니다. 실행 단계에서 `.search()`나 LLM을 다시 호출하지 않습니다.

표시 상한: 50행·100노드·150관계, 경로는 고정 길이 또는 최대 3홉. 50행에 닿으면 전체 결과인지 확정하지 않고 잘림 가능성을 알립니다. 최종 LIMIT을 자동으로 붙이거나 UNION의 의미를 바꾸지 않습니다. 지원 문법·함수 밖이면 실행을 거부합니다.

In [4]:
state = {"generated": None, "validation": None, "execution": None, "question": None}
question_input = widgets.Textarea(value="보호자 연락 행동과 근거 문서를 그래프로 보여줘", description="질문", layout=widgets.Layout(width="98%", height="75px"))
cypher_input = widgets.Textarea(value="", description="생성문", layout=widgets.Layout(width="98%", height="160px"))
browser_copy = widgets.HTML(value="<p>Aura 복사: 검사를 통과하면 쿼리가 표시됩니다.</p>")

def set_browser_copy(text):
    browser_copy.value = "<label>Aura 복사</label><textarea readonly style='width:98%;height:160px'>" + escape(text) + "</textarea>"
generate_button = widgets.Button(description="1. Cypher 생성", disabled=not ready)
validate_button = widgets.Button(description="2. 실행 전 검사", disabled=True)
execute_button = widgets.Button(description="3. 조회 실행", disabled=True)
output = widgets.Output()

def log_event(event, extra):
    record = {"case_id": "INTERACTIVE", "event": event, "run_at_kst": demo.now(),
              "question": state["question"], "schema_version": schema["schema_version"],
              "model": llm.model_name, "libraries": demo.versions(),
              "generated_cypher": state["generated"]["generated_cypher"] if state["generated"] else None,
              **extra}
    with (folder / "text2cypher_eval.jsonl").open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

def invalidate_query(change):
    state["validation"] = None
    state["execution"] = None
    execute_button.disabled = True
    set_browser_copy("")

def invalidate_question(change):
    invalidate_query(change)
    state["generated"] = None
    validate_button.disabled = True

cypher_input.observe(invalidate_query, names="value")
question_input.observe(invalidate_question, names="value")

def on_generate(_):
    with output:
        output.clear_output()
        state.update(generated=None, validation=None, execution=None, question=question_input.value)
        validate_button.disabled = execute_button.disabled = True
        try:
            generated = demo.generate_cypher(state["question"], schema, llm)
            state["generated"] = generated
            cypher_input.value = generated["generated_cypher"]
            validate_button.disabled = False
            log_event("GENERATED", {"generation_seconds": generated["generation_seconds"], "execution_status": "NOT_EXECUTED"})
            print("생성 완료. 아직 생성 쿼리는 DB에서 실행하지 않았습니다.")
        except Exception as error:  # noqa: BLE001 -- 접속 정보를 출력하지 않음
            log_event("GENERATION_ERROR", {"error": demo.safe_error(error)})
            display(JSON({"status": "GENERATION_ERROR", "error": demo.safe_error(error)}))

def on_validate(_):
    with output:
        output.clear_output()
        validation = demo.validate_cypher(driver, database, cypher_input.value, schema)
        cypher_input.value = validation["cypher"]
        state["validation"] = validation
        execute_button.disabled = validation["status"] != "VALID"
        set_browser_copy(demo.prepare_browser_query(validation) if validation["status"] == "VALID" else "")
        log_event("VALIDATED", {"validation": validation, "execution_status": "NOT_EXECUTED"})
        display(JSON(validation))

def on_execute(_):
    global raw_records
    with output:
        output.clear_output()
        validation = state["validation"]
        if validation is None or cypher_input.value != validation["cypher"]:
            print("쿼리가 변경됐습니다. 실행 전 검사를 다시 하세요.")
            return
        try:
            execution = demo.execute_cypher(driver, database, validation, schema)
            state["execution"] = execution
            raw_records = execution["records"]
            payload = demo.serialize_records(raw_records)
            log_event("EXECUTED", {"executed_cypher": execution["executed_cypher"], "validation_status": validation["status"],
                                  "execution_seconds": execution["execution_seconds"], "row_count": len(raw_records),
                                  "truncated": execution["truncated"], "possibly_truncated": execution["possibly_truncated"],
                                  "payload": payload, "comparison_pass": None})
            display(JSON({"status": "OK" if raw_records else "NO_RESULT", "row_count": len(raw_records),
                          "node_count": payload["graph"]["node_count"], "relationship_count": payload["graph"]["relationship_count"],
                          "possibly_truncated": execution["possibly_truncated"], "graph_display_truncated": payload["graph"]["truncated"]}))
            if payload["graph"]["nodes"]:
                print("Path 객체는 raw_records에 보존했습니다. Aura 복사란의 쿼리를 같은 프로젝트 DB에서 실행해 그래프 보기로 전환하세요.")
                display(JSON(payload["graph"]))
            else:
                display(JSON(payload["rows"]))
            print("Text2Cypher 결과는 근거 인용(Citation)을 대신하지 않음")
        except Exception as error:  # noqa: BLE001 -- 접속 정보가 포함될 수 있는 오류 원문 생략
            log_event("EXECUTION_ERROR", {"error": demo.safe_error(error), "executed_cypher": validation["cypher"]})
            display(JSON({"status": "EXECUTION_ERROR", "error": demo.safe_error(error)}))

generate_button.on_click(on_generate)
validate_button.on_click(on_validate)
execute_button.on_click(on_execute)
display(question_input, widgets.HBox([generate_button, validate_button, execute_button]), cypher_input, output, browser_copy)

Textarea(value='보호자 연락 행동과 근거 문서를 그래프로 보여줘', description='질문', layout=Layout(height='75px', width='98%'))

Textarea(value='', description='생성문', layout=Layout(height='160px', width='98%'))

Output()

HTML(value='<p>Aura 복사: 검사를 통과하면 쿼리가 표시됩니다.</p>')

## 4. 실제 실행 평가 기록 보기

예시 2문항과 새 조건·별칭·문서 구조·집계·없는 대상·쓰기 요청 6문항을 구분했습니다. 조회 6문항은 사람이 쓴 방식의 기준 쿼리와 **ID 집합·방향·집계값**을 비교했고, 나머지 2문항은 실행 전 거부를 확인했습니다. 별도 사람 판독을 마친 품질 점수나 일반 성능으로 해석하지 않습니다.

아래는 저장된 실행 결과의 표시이며 새로운 LLM 호출이 아닙니다. 재평가는 저장소 루트에서 `uv run --no-sync python notebooks/respiratory/text2cypher_evaluate.py`로 수행합니다.

In [5]:
summary_path = folder / "text2cypher_eval_summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    display(JSON(summary))
    saved = [json.loads(line) for line in (folder / "text2cypher_eval.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
    by_id = {row["case_id"]: row for row in saved if row.get("case_id") in {"E01", "E02", "N01", "N02", "N03", "N04", "N05", "N06"}}
    display(JSON([{key: row.get(key) for key in ("case_id", "question", "seen_example", "generated_cypher", "status", "row_count", "comparison_pass", "total_seconds")} for _, row in sorted(by_id.items())]))
else:
    print("실제 실행 평가 파일이 없습니다. 평가 스크립트를 먼저 실행하세요.")

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

## 5. 같은 Aura 화면에서 그래프 확인

1. 팀 Aura 콘솔에서 **프로젝트 인스턴스**를 선택합니다. 개인 인스턴스나 로컬 DB로 바꾸지 않습니다.
2. 해당 인스턴스의 Query/Browser 조회 화면을 열고, 위 연결 셀과 같은 데이터베이스를 선택합니다. 실제 화면 명칭은 사용자 환경에서 확인합니다.
3. 위 **Aura 복사**란의 검증된 Cypher를 변경하지 않고 붙여 실행합니다. 쿼리에는 Python 전용 변수·누락 파라미터가 없습니다.
4. 그래프 질문이면 Graph 보기, 집계 질문이면 Table 보기를 선택합니다. Python 조회만으로 웹 화면이 자동 갱신되지는 않습니다.
5. Document·Chunk를 선택해 document_id·PDF 쪽수·원문을, 관계에서 타입·방향·DONE_BY의 basis를 확인합니다. embedding을 프롬프트·로그로 복사하지 않습니다.
6. 쿼리를 수정했다면 수정 전후를 보존하고 Python에서도 같은 수정문을 재검사·실행합니다.

아래 확인은 **사용자의 실제 Aura 화면 확인 기록**입니다. 자동 검증이 아니며 확인 전에는 체크하지 않습니다.

In [6]:
same_instance = widgets.Checkbox(value=False, description="프로젝트 인스턴스·DB가 위 연결과 같음")
same_query = widgets.Checkbox(value=False, description="Aura 복사란의 동일 쿼리를 실행함")
graph_seen = widgets.Checkbox(value=False, description="그래프 보기에서 노드·관계·문서·쪽수를 확인함")
observed_rows = widgets.IntText(value=0, description="Aura 행 수")
confirm = widgets.Button(description="실제 확인 기록 저장")
confirmation_output = widgets.Output()

def confirm_browser(_):
    with confirmation_output:
        confirmation_output.clear_output()
        execution = state["execution"]
        validation = state["validation"]
        if execution is None or validation is None or not all([same_instance.value, same_query.value, graph_seen.value]):
            print("실제 조회 실행과 세 확인 항목이 모두 필요합니다.")
            return
        if observed_rows.value != len(execution["records"]):
            print("Python과 Aura 행 수가 다릅니다. 같은 쿼리·DB·데이터 상태인지 확인하세요.")
            return
        proof = {"status": "USER_CONFIRMED", "source": "notebook manual confirmation", "confirmed_at_kst": demo.now(),
                 "instance_fingerprint": connection["instance_fingerprint"], "database": connection["database"],
                 "cypher_sha256": hashlib.sha256(execution["executed_cypher"].encode()).hexdigest(),
                 "row_count": observed_rows.value, "not_automated_visual_verification": True}
        schema["browser_verification"] = proof
        demo.SCHEMA_PATH.write_text(json.dumps(schema, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        log_event("BROWSER_USER_CONFIRMED", {"browser_verification": proof})
        display(JSON(proof))

confirm.on_click(confirm_browser)
display(same_instance, same_query, graph_seen, observed_rows, confirm, confirmation_output)

Checkbox(value=False, description='프로젝트 인스턴스·DB가 위 연결과 같음')

Checkbox(value=False, description='Aura 복사란의 동일 쿼리를 실행함')

Checkbox(value=False, description='그래프 보기에서 노드·관계·문서·쪽수를 확인함')

IntText(value=0, description='Aura 행 수')

Button(description='실제 확인 기록 저장', style=ButtonStyle())

Output()

In [7]:
# 자동 노트북 검증 때만 UI 콜백을 실행한다. 일반 재실행에서는 LLM을 호출하지 않는다.
import os

if os.getenv("TEXT2CYPHER_NOTEBOOK_SMOKE") == "1":
    assert ready
    question_input.value = "보호자 연락 행동과 근거 문서를 그래프로 보여줘"
    on_generate(None)
    assert state["generated"] is not None
    on_validate(None)
    assert state["validation"]["status"] == "VALID"
    on_execute(None)
    assert state["execution"] is not None
    assert len(state["execution"]["records"]) == 8
    assert state["execution"]["executed_cypher"] == state["validation"]["cypher"]
    print("노트북 UI 실제 실행 확인: OK, 8행, 생성·검사·실행 쿼리 일치")

노트북 UI 실제 실행 확인: OK, 8행, 생성·검사·실행 쿼리 일치


## 종료

DB 연결을 닫습니다. 아래 셀을 실행한 뒤 다시 조회하려면 연결 셀과 입력 UI 셀을 다시 실행합니다. 비밀 설정을 출력하거나 노트북에 저장하지 않습니다.

In [8]:
# 시연을 마친 뒤에만 실행하세요. 전체 셀 실행으로 입력 UI가 닫히지 않도록 기본값은 False입니다.
close_connection = False
if close_connection and driver is not None:
    driver.close()
    ready = False
    generate_button.disabled = validate_button.disabled = execute_button.disabled = True
    print("DB 연결을 닫았습니다.")